### 00 LIBRERIAS

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob
# Aplicar configuraciones de visualización total de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

from pathlib import Path
import glob

import warnings
warnings.filterwarnings("ignore")
import plotly.express as px

import func_negocio as func

### 01 EXTRACT

In [3]:
bd_locales = r"input\Productividad 2026 v4 --- ep.xlsx"
df_locales = pd.read_excel(bd_locales, sheet_name="Locales")

In [4]:
bd_2025 = r"input\bd_ventas\Export_VentaProdBI_2025.xlsx"
df_2025 = pd.read_excel(bd_2025)


df_2025["codigo_local"] = df_2025["Codigo Local"]
df_2025["tipo_venta"] = df_2025["Tipo Venta"]
df_2025["nombre_area"] = df_2025["nombre_area"]
df_2025["nombre_division"] = df_2025["nombre_division"]
df_2025["VtaNeta"] = df_2025["VtaNeta"]
df_2025["VtaUnidad"] = df_2025["VtaUnidad"]
df_2025["VtaNetaPpto"] = df_2025["VtaNetaPpto"]

df_2025 = df_2025[~df_2025["nombre_division"].isnull()]

In [5]:
lista_archivos = [
    "Export_VentaProdBI_202601.xlsx",
    "Export_VentaProdBI_202602.xlsx",
    "Export_VentaProdBI_202603.xlsx",
    "Export_EstimarMesActual_20260430.xlsx",
    "Export_EstimarMesActual_202605.xlsx",
    "Export_EstimarMesActual_20260611.xlsx",
]
ruta = r"input\bd_ventas"

rutas_completas = [os.path.join(ruta, archivo) for archivo in lista_archivos]
dfs = [pd.read_excel(p) for p in rutas_completas]
df_2026 = pd.concat(dfs, ignore_index=True)

df_2026["codigo_local"] = df_2026["Codigo Local"]
df_2026["tipo_venta"] = df_2026["Tipo Venta"]
df_2026["nombre_area"] = df_2026["nombre_area"]
df_2026["nombre_division"] = df_2026["nombre_division"]
df_2026["VtaNeta"] = df_2026["VtaNeta"]
df_2026["VtaUnidad"] = df_2026["VtaUnidad"]
df_2026["VtaNetaPpto"] = df_2026["VtaNetaPpto"]

df_2026 = df_2026[~df_2026["nombre_division"].isnull()]
df_2026["Anio"] = df_2026["Anio"].fillna(df_2026["Fecha"].dt.year)
df_2026["NroMes"] = df_2026["NroMes"].fillna(df_2026["Fecha"].dt.month)


In [6]:
df_ventas_all = pd.concat([df_2025, df_2026])


In [7]:
bd_ppto_2026 = r"input\bd_ventas\Export_PptoVenta.xlsx"
df_ppto_2026 = pd.read_excel(bd_ppto_2026)

df_ppto_2026["codigo_local"] = df_ppto_2026["codigo_local"]
df_ppto_2026["tipo_venta"] = df_ppto_2026["DpPv_Lv4"]
df_ppto_2026["nombre_area"] = df_ppto_2026["Area"]
df_ppto_2026["nombre_division"] = df_ppto_2026["Division"]
df_ppto_2026["VtaNeta"] = df_ppto_2026["Ppto"]
df_ppto_2026["VtaUnidad"] = df_ppto_2026["Ppto"]
df_ppto_2026["VtaNetaPpto"] = df_ppto_2026["Ppto"]
df_ppto_2026["Anio"] = df_ppto_2026["Periodo"].astype(str).str[0:4]
df_ppto_2026["NroMes"] = df_ppto_2026["Periodo"].astype(str).str[-2:]

df_ppto_2026 = df_ppto_2026[~df_ppto_2026["nombre_division"].isnull()]
df_ppto_2026["Anio"] = df_ppto_2026["Anio"].astype(int)
df_ppto_2026["NroMes"] = df_ppto_2026["NroMes"].astype(int)

### 02 TRANSFORM

In [8]:
df_ventas_all["codigo_local"] = df_ventas_all["codigo_local"].astype(str)
df_locales["codigo_local"] = df_locales["codigo_local"].astype(str)

df_2025x = df_ventas_all.merge(df_locales[["codigo_local", "_Tienda"]], on="codigo_local", how="inner")
df_2025x["area"] = np.where(df_2025x["nombre_area"] == "ELECTRO", "ELECTRO",
                           np.where(df_2025x["nombre_division"]== "FRESCOS", "FRESCOS",
                                    np.where(df_2025x["nombre_division"]=="NON FOOD", "NONFOOD", "ABARROTES")))

df_ventas_final = func.logica_area_puesto( df_2025x, func.config_reglas_prod)

In [9]:
df_ppto_2026["codigo_local"] = df_ppto_2026["codigo_local"].astype(str)
df_locales["codigo_local"] = df_locales["codigo_local"].astype(str)

df_ppto_2026x = df_ppto_2026.merge(df_locales[["codigo_local", "_Tienda"]], on="codigo_local", how="inner")
df_ppto_2026x["area"] = np.where(df_ppto_2026x["nombre_area"] == 'ELECTRO A11', "ELECTRO",
                           np.where(df_ppto_2026x["nombre_division"]== 'FRESCOS D2', "FRESCOS",
                                    np.where(df_ppto_2026x["nombre_division"]=='NON FOOD D3', "NONFOOD", "ABARROTES")))

df_ppto_2026x["tipo_venta"] = np.where(df_ppto_2026x["tipo_venta"]=='E-Commerce', "Venta Ecommerce",
                                      np.where(df_ppto_2026x["tipo_venta"]=='Tienda', "Venta POS",
                                               np.where(df_ppto_2026x["tipo_venta"]=='Last Mile', "Last Mile",df_ppto_2026x["tipo_venta"] )))

df_ppto_2026xx = func.logica_area_puesto( df_ppto_2026x, func.config_reglas_prod)
df_ppto_2026xx["Ppto2026"] = df_ppto_2026xx["Vta."]
df_ppto_2026z = df_ppto_2026xx[['_Tienda', 'Anio', 'NroMes', 'area','Ppto2026']]

### 03 LOAD

In [10]:
df_ventas_final["_Periodo"] = df_ventas_final["Anio"].astype(int).astype(str)+df_ventas_final["NroMes"].astype(int).astype(str).str.zfill(2)
df_ventas_final = df_ventas_final.rename(columns={"area": "_Tipo"})

In [11]:
df_ppto_2026z["_Periodo"] = df_ppto_2026z["Anio"].astype(int).astype(str)+df_ppto_2026z["NroMes"].astype(int).astype(str).str.zfill(2)
df_ppto_2026z = df_ppto_2026z.rename(columns={"area": "_Tipo"})

In [12]:
df_ppto_2026z["_Tipo"].unique()

array(['ELECTRO', 'FRESCOS', 'ALMACEN FRESCOS', 'ABARROTES', 'CAJAS',
       'RECEPCION', 'ALMACEN', 'INVENTARIOS', 'MULTIFUNCIONAL'],
      dtype=object)

In [13]:
df_ventas_final.to_parquet(r"output\VentasReal.parquet")
df_ppto_2026z.to_parquet(r"output\VentasPPTO.parquet")

In [1]:
import pandas as pd
import numpy as np

# 1. Creación del DataFrame con los datos de la imagen
data = {
    'GOR': ['Joe H', 'Melany A', 'José A', 'Rodolfo O', 'Vik E', 'Johanna V', 'Daniel R', 'Fátima L', 'María R'],
    'Productividad': [104362, 98866, 98852, 98539, 93516, 92834, 82086, 67320, 53110],
    'Venta': [444644259, 417894386, 430547365, 534588438, 620128807, 380822947, 341369949, 56380933, 130351279],
    'Ene': [-0.02, 0.15, -0.05, 0.01, -0.01, -0.07, -0.01, -0.15, -0.10],
    'Feb': [0.03, 0.22, -0.04, 0.04, 0.01, -0.09, -0.01, 0.00, -0.15],
    'Mar': [-0.03, 0.06, -0.05, -0.05, -0.02, -0.05, 0.00, 0.04, -0.23],
    'Abr': [0.07, 0.17, 0.05, 0.10, 0.08, 0.05, 0.10, 0.04, -0.21],
    'May': [0.04, 0.14, 0.04, 0.08, 0.06, 0.03, 0.11, 0.08, -0.22],
    'Jun': [0.06, 0.17, 0.06, 0.08, 0.04, 0.06, 0.13, 0.14, -0.12],
    'Total': [0.02, 0.15, 0.00, 0.04, 0.03, -0.02, 0.05, 0.02, -0.18]
}

df = pd.DataFrame(data)

# 2. Definición de estilos CSS para replicar la estética (Cabecera negra, texto alineado)
headers_props = [
    ('background-color', '#111111'),
    ('color', 'white'),
    ('font-weight', 'bold'),
    ('text-align', 'center')
]
cell_props = [('text-align', 'center'), ('font-family', 'Arial')]

# Listas de columnas por tipo para aplicar formatos vectorizados
meses_cols = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Total']

# 3. Pipeline de estilización (Vectorizado vía Styler)
styled_df = (df.style
    .set_table_styles([
        {'selector': 'th', 'props': headers_props},
        {'selector': 'td', 'props': cell_props}
    ])
    .hide(axis='index') # Oculta el índice por defecto de pandas
    
    # Formateo de números sin bucles
    .format({'Productividad': '{:,.0f}', 'Venta': '{:,.0f}'})
    .format({col: '{:+.02%}' for col in meses_cols}) # Añade el signo + o - explícito
    
    # Gráficos de barras integrados (Data bars)
    .bar(subset=['Productividad'], color='#8bb1dc', vmin=0, cmap=None)
    .bar(subset=['Venta'], color='#fbc55e', vmin=0, cmap=None)
    
    # Mapa de calor condicional (Divergente: Rojo - Blanco/Crema - Verde)
    .background_gradient(subset=meses_cols, cmap='RdYlGn', vmin=-0.25, vmax=0.25, axis=None)
)

# 4. Renderizar o guardar el resultado
# Si estás en Jupyter Notebook, basta con ejecutar: styled_df
# Para guardarlo como archivo HTML:
styled_df.to_html('reporte_gor.html')

In [2]:
import pandas as pd
import numpy as np

# 1. Creación del DataFrame con los datos de la imagen
data = {
    'GOR': ['Joe H', 'Melany A', 'José A', 'Rodolfo O', 'Vik E', 'Johanna V', 'Daniel R', 'Fátima L', 'María R'],
    'Productividad': [104362, 98866, 98852, 98539, 93516, 92834, 82086, 67320, 53110],
    'Venta': [444644259, 417894386, 430547365, 534588438, 620128807, 380822947, 341369949, 56380933, 130351279],
    'Ene': [-0.02, 0.15, -0.05, 0.01, -0.01, -0.07, -0.01, -0.15, -0.10],
    'Feb': [0.03, 0.22, -0.04, 0.04, 0.01, -0.09, -0.01, 0.00, -0.15],
    'Mar': [-0.03, 0.06, -0.05, -0.05, -0.02, -0.05, 0.00, 0.04, -0.23],
    'Abr': [0.07, 0.17, 0.05, 0.10, 0.08, 0.05, 0.10, 0.04, -0.21],
    'May': [0.04, 0.14, 0.04, 0.08, 0.06, 0.03, 0.11, 0.08, -0.22],
    'Jun': [0.06, 0.17, 0.06, 0.08, 0.04, 0.06, 0.13, 0.14, -0.12],
    'Total': [0.02, 0.15, 0.00, 0.04, 0.03, -0.02, 0.05, 0.02, -0.18]
}

df = pd.DataFrame(data)

# 2. Definición de estilos CSS para replicar la estética (Cabecera negra, texto alineado)
headers_props = [
    ('background-color', '#111111'),
    ('color', 'white'),
    ('font-weight', 'bold'),
    ('text-align', 'center')
]
cell_props = [('text-align', 'center'), ('font-family', 'Arial')]

# Listas de columnas por tipo para aplicar formatos vectorizados
meses_cols = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Total']

# 3. Pipeline de estilización (Vectorizado vía Styler)
styled_df = (df.style
    .set_table_styles([
        {'selector': 'th', 'props': headers_props},
        {'selector': 'td', 'props': cell_props}
    ])
    .hide(axis='index') # Oculta el índice por defecto de pandas
    
    # Formateo de números sin bucles
    .format({'Productividad': '{:,.0f}', 'Venta': '{:,.0f}'})
    .format({col: '{:+.02%}' for col in meses_cols}) # Añade el signo + o - explícito
    
    # Gráficos de barras integrados (Data bars)
    .bar(subset=['Productividad'], color='#8bb1dc', vmin=0, cmap=None)
    .bar(subset=['Venta'], color='#fbc55e', vmin=0, cmap=None)
    
    # Mapa de calor condicional (Divergente: Rojo - Blanco/Crema - Verde)
    .background_gradient(subset=meses_cols, cmap='RdYlGn', vmin=-0.25, vmax=0.25, axis=None)
)

# 4. Renderizar o guardar el resultado
# Si estás en Jupyter Notebook, basta con ejecutar: styled_df
# Para guardarlo como archivo HTML:
styled_df.to_html('reporte_gor.html')

In [4]:
import pandas as pd
import numpy as np

# 1. Datos (los mismos de la imagen anterior)
data = {
    'GOR': ['Joe H', 'Melany A', 'José A', 'Rodolfo O', 'Vik E', 'Johanna V', 'Daniel R', 'Fátima L', 'María R'],
    'Productividad': [104362, 98866, 98852, 98539, 93516, 92834, 82086, 67320, 53110],
    'Venta': [444644259, 417894386, 430547365, 534588438, 620128807, 380822947, 341369949, 56380933, 130351279],
    'Ene': [-0.02, 0.15, -0.05, 0.01, -0.01, -0.07, -0.01, -0.15, -0.10],
    'Feb': [0.03, 0.22, -0.04, 0.04, 0.01, -0.09, -0.01, 0.00, -0.15],
    'Mar': [-0.03, 0.06, -0.05, -0.05, -0.02, -0.05, 0.00, 0.04, -0.23],
    'Abr': [0.07, 0.17, 0.05, 0.10, 0.08, 0.05, 0.10, 0.04, -0.21],
    'May': [0.04, 0.14, 0.04, 0.08, 0.06, 0.03, 0.11, 0.08, -0.22],
    'Jun': [0.06, 0.17, 0.06, 0.08, 0.04, 0.06, 0.13, 0.14, -0.12],
    'Total': [0.02, 0.15, 0.00, 0.04, 0.03, -0.02, 0.05, 0.02, -0.18]
}

df = pd.DataFrame(data)
meses_cols = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Total']

# 2. Configuración de Estilos CSS Avanzados para la Estética

# Estilo general de la tabla (fuente, bordes, espaciado)
table_styles = [
    {'selector': '', 'props': [
        ('font-family', '"Helvetica Neue", Helvetica, Arial, sans-serif'),
        ('border-collapse', 'collapse'),
        ('width', '100%'),
        ('margin', '20px 0'),
        ('font-size', '14px'),
        ('color', '#333')
    ]},
    # Estilo de las cabeceras (fondo oscuro suave, texto bold, padding)
    {'selector': 'th', 'props': [
        ('background-color', '#2c3e50'), # Un gris azulado más suave que el negro puro
        ('color', 'white'),
        ('font-weight', '600'),
        ('text-align', 'center'),
        ('padding', '12px 8px'),
        ('border', 'none'),
        ('text-transform', 'uppercase'),
        ('letter-spacing', '0.05em')
    ]},
    # Estilo de las celdas (padding para que respiren, alineación)
    {'selector': 'td', 'props': [
        ('padding', '10px 8px'),
        ('border-bottom', '1px solid #eee'), # Líneas divisorias muy sutiles
        ('text-align', 'right'), # Alineación a la derecha para números
        ('border-right', 'none'),
        ('border-left', 'none')
    ]},
    # Alineación a la izquierda específica para la columna GOR
    {'selector': '.col0', 'props': [('text-align', 'left')]} 
]

# 3. Pipeline de Estilización Mejorado

styled_df = (df.style
    # Aplicar los estilos CSS base
    .set_table_styles(table_styles)
    .hide(axis='index')
    
    # Formateo numérico
    .format({'Productividad': '{:,.0f}', 'Venta': '{:,.0f}'})
    .format({col: '{:+.0%}' for col in meses_cols}) # Formato de porcentaje limpio
    
    # --- BARRAS DE DATOS PROFESIONALES ---
    # Mejoramos las barras: definimos anchos fijos y colores más suaves.
    # 'Productividad' - Barra Azul Sutil
    .bar(subset=['Productividad'], color='#a9cce3', vmin=0, align='mid', 
         props="width: 120px; border-radius: 4px; border-right: 1px solid white;")
    # 'Venta' - Barra Amarilla/Naranja Suave
    .bar(subset=['Venta'], color='#f9e79f', vmin=0, align='mid',
         props="width: 140px; border-radius: 4px; border-right: 1px solid white;")
    
    # --- MAPA DE CALOR SOFISTICADO ---
    # Usamos una paleta divergente 'PiYG' (Pink-Yellow-Green) o 'PRGn' 
    # que son más estéticas que la estándar 'RdYlGn'.
    .background_gradient(subset=meses_cols, cmap='PRGn', vmin=-0.25, vmax=0.25, axis=None)
    
    # --- DETALLES DE TEXTO ---
    # Destacamos la columna GOR
    .set_properties(subset=['GOR'], **{'font-weight': 'bold', 'color': '#2c3e50'})
)

# 4. Renderizar o guardar
#styled_df # Si estás en Jupyter, ejecuta esta línea
styled_df.to_html('reporte_gor_estetico.html')